
# 01 — Data Preparation

## Deep Learning in Asset Pricing — Replication Project

**Reference:** Chen, L., Pelger, M., & Zhu, J. (2023), *Deep Learning in Asset Pricing*, Management Science, 70(2), 714–750.

### Purpose of this notebook

This notebook builds the reproducible data pipeline used by the rest of the replication project. It is intentionally separated from model training so that every later model—linear benchmark, feedforward network, LSTM, and GAN—uses exactly the same cleaned and time-split inputs.

### Main tasks

1. Load the stock-return / firm-characteristic panel.
2. Load the macroeconomic time series.
3. Identify and validate the date, stock identifier, return, and characteristic fields.
4. Reproduce the paper-style complete-case stock sample.
5. Cross-sectionally rank-normalize firm characteristics.
6. Apply the fixed train / validation / test split.
7. Run data-quality checks.
8. Save processed datasets for later notebooks.

> **Important:** This notebook does not train an asset-pricing model. Its only job is to make the input data correct, transparent, and reproducible.



## 1. Replication targets from the paper

The paper's empirical U.S. equity application uses:

- Monthly U.S. stock returns from **January 1967 through December 2016**.
- **46 time-varying firm characteristics**.
- **178 macroeconomic time series**.
- Stocks with complete firm-characteristic information in a given month for the core empirical sample.
- Cross-sectional rank / quantile normalization of firm characteristics.
- A fixed chronological split:
  - **Training:** 1967–1986
  - **Validation:** 1987–1991
  - **Test:** 1992–2016

The macroeconomic information described in the paper combines FRED-MD variables, cross-sectional characteristic medians, and additional equity-premium predictors. If the supplied `Macro.csv` is already the authors' processed macro file, this notebook will preserve it rather than re-transforming the series.

### Expected local files

For this replication, place the author-provided or project-provided files in:

```text
data/raw/
```

The default filenames used below are:

```text
RetChar.csv
Macro.csv
```

If your files have different names, change only the configuration cell below.


In [ ]:

# 2. Imports and reproducibility

from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("NumPy :", np.__version__)
print("pandas:", pd.__version__)
print("Seed  :", SEED)


In [ ]:

# 3. Locate the repository root robustly

def find_repo_root(start: Path | None = None) -> Path:
    """Search upward for the project root using requirements.txt or .git."""
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "requirements.txt").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Open Jupyter from inside the cloned project repository."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
NOTEBOOK_DIR = ROOT / "notebooks"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root :", ROOT)
print("Raw data         :", RAW_DIR)
print("Processed data   :", PROCESSED_DIR)


In [ ]:

# 4. Configuration

RETCHAR_FILE = "RetChar.csv"
MACRO_FILE = "Macro.csv"

# Core paper-style sample treatment:
STRICT_PAPER_REPLICATION = True

# Fixed chronological split
TRAIN_START = pd.Timestamp("1967-01-01")
TRAIN_END   = pd.Timestamp("1986-12-31")

VALID_START = pd.Timestamp("1987-01-01")
VALID_END   = pd.Timestamp("1991-12-31")

TEST_START  = pd.Timestamp("1992-01-01")
TEST_END    = pd.Timestamp("2016-12-31")

print("Configured stock file:", RETCHAR_FILE)
print("Configured macro file:", MACRO_FILE)
print("Strict paper replication:", STRICT_PAPER_REPLICATION)


In [ ]:

# 5. Inventory the raw-data directory

raw_files = sorted([p for p in RAW_DIR.iterdir() if p.is_file() and p.name != ".gitkeep"])

if raw_files:
    display(pd.DataFrame({
        "file": [p.name for p in raw_files],
        "size_MB": [round(p.stat().st_size / 1024**2, 3) for p in raw_files],
    }))
else:
    print(
        "No raw data files found yet.\n"
        f"Place {RETCHAR_FILE} and {MACRO_FILE} in:\n{RAW_DIR}"
    )


In [ ]:

# 6. Generic tabular loader

def load_table(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"[MISSING] {path.name}")
        return None

    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    print(f"[LOADED] {path.name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df


retchar_raw = load_table(RAW_DIR / RETCHAR_FILE)
macro_raw = load_table(RAW_DIR / MACRO_FILE)



## 7. Standardize the stock / characteristic panel

The original input is expected to contain one row per stock-month with:

- a date,
- a stock identifier such as `permno`,
- a monthly return such as `ret`,
- and 46 firm characteristics.

The helper functions below detect common naming variants. We still print the detected fields so that the mapping is explicit and can be corrected if necessary.


In [ ]:

# 7A. Column-name and date helpers

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in out.columns
    ]
    return out


def detect_column(columns, candidates, label):
    cols = list(columns)
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(
        f"Could not detect the {label} column. "
        f"Tried {candidates}. Available columns begin with: {cols[:20]}"
    )


def parse_monthly_date(s: pd.Series) -> pd.Series:
    """Parse common monthly date representations, including YYYYMM integers."""
    s = s.copy()

    if pd.api.types.is_numeric_dtype(s):
        text = s.astype("Int64").astype(str).str.replace("<NA>", "", regex=False)
    else:
        text = s.astype(str).str.strip()

    # First try YYYYMM exactly.
    yyyymm_mask = text.str.fullmatch(r"\d{6}")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    if yyyymm_mask.any():
        out.loc[yyyymm_mask] = pd.to_datetime(
            text.loc[yyyymm_mask], format="%Y%m", errors="coerce"
        )

    # Parse the rest generically.
    remaining = ~yyyymm_mask
    if remaining.any():
        out.loc[remaining] = pd.to_datetime(
            text.loc[remaining], errors="coerce"
        )

    # Canonicalize to month start.
    return out.dt.to_period("M").dt.to_timestamp()


In [ ]:

# 7B. Normalize and inspect the stock panel

if retchar_raw is not None:
    retchar = clean_column_names(retchar_raw)

    DATE_COL = detect_column(
        retchar.columns,
        ["date", "yyyymm", "month", "caldt", "time"],
        "date",
    )
    ID_COL = detect_column(
        retchar.columns,
        ["permno", "id", "stock_id", "security_id", "permco"],
        "stock identifier",
    )
    RET_COL = detect_column(
        retchar.columns,
        ["ret", "return", "ret_excess", "excess_ret", "retx"],
        "return",
    )

    retchar[DATE_COL] = parse_monthly_date(retchar[DATE_COL])
    retchar[RET_COL] = pd.to_numeric(retchar[RET_COL], errors="coerce")

    NON_CHARACTERISTIC_COLS = {
        DATE_COL, ID_COL, RET_COL,
        "sic2", "sic", "exchcd", "shrcd",
        "mktcap", "market_cap", "me",
    }

    characteristic_cols = [
        c for c in retchar.columns
        if c not in NON_CHARACTERISTIC_COLS
        and pd.api.types.is_numeric_dtype(retchar[c])
    ]

    print("Detected date column      :", DATE_COL)
    print("Detected stock ID column  :", ID_COL)
    print("Detected return column    :", RET_COL)
    print("Numeric characteristic candidates:", len(characteristic_cols))
    print("Date range:", retchar[DATE_COL].min(), "to", retchar[DATE_COL].max())
    print("Unique stocks:", f"{retchar[ID_COL].nunique():,}")

    display(retchar.head())
else:
    print("Skipped: stock panel has not been loaded.")


In [ ]:

# 8. Stock-panel data quality report

if retchar_raw is not None:
    stock_quality = pd.Series({
        "rows": len(retchar),
        "columns": retchar.shape[1],
        "unique_stocks": retchar[ID_COL].nunique(),
        "unique_months": retchar[DATE_COL].nunique(),
        "duplicate_stock_months": retchar.duplicated([DATE_COL, ID_COL]).sum(),
        "missing_dates": retchar[DATE_COL].isna().sum(),
        "missing_stock_ids": retchar[ID_COL].isna().sum(),
        "missing_returns": retchar[RET_COL].isna().sum(),
        "characteristic_candidates": len(characteristic_cols),
    }, name="value")

    display(stock_quality.to_frame())

    char_missing = (
        retchar[characteristic_cols]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_fraction")
        .to_frame()
    )
    display(char_missing.head(20))

    if len(characteristic_cols) != 46:
        print(
            f"WARNING: detected {len(characteristic_cols)} numeric characteristic candidates, "
            "while the paper uses 46. Review NON_CHARACTERISTIC_COLS and the source-file schema."
        )
else:
    print("Skipped: stock panel has not been loaded.")



## 9. Complete-case filtering and cross-sectional rank normalization

For the strict paper-style replication, we retain stock-month observations only when all selected firm characteristics are observed.

We then transform each characteristic within each month into a cross-sectional percentile rank and center it to approximately **[-0.5, 0.5]**:

\[
\widetilde{x}_{i,t}
=
\operatorname{rankpct}_{t}(x_{i,t}) - 0.5.
\]

This removes scale differences across characteristics and makes a stock's cross-sectional position the relevant input.

> We do **not** silently median-impute missing firm characteristics in strict replication mode.


In [ ]:

# 9A. Filter to complete cases

if retchar_raw is not None:
    before_rows = len(retchar)

    required_cols = [DATE_COL, ID_COL, RET_COL] + characteristic_cols

    stock_clean = retchar.dropna(subset=[DATE_COL, ID_COL, RET_COL]).copy()

    if STRICT_PAPER_REPLICATION:
        stock_clean = stock_clean.dropna(subset=characteristic_cols).copy()

    after_rows = len(stock_clean)

    print(f"Rows before filtering : {before_rows:,}")
    print(f"Rows after filtering  : {after_rows:,}")
    print(f"Rows retained         : {after_rows / before_rows:.2%}" if before_rows else "n/a")

    if characteristic_cols:
        remaining_char_missing = stock_clean[characteristic_cols].isna().sum().sum()
        print("Remaining missing characteristic cells:", int(remaining_char_missing))
else:
    print("Skipped: stock panel has not been loaded.")


In [ ]:

# 9B. Cross-sectional rank normalization to approximately [-0.5, 0.5]

def rank_normalize_cross_section(
    df: pd.DataFrame,
    date_col: str,
    columns: list[str],
) -> pd.DataFrame:
    out = df.copy()

    # pct=True creates ranks in (0, 1]; subtracting 0.5 centers the distribution.
    out[columns] = (
        out.groupby(date_col, observed=True)[columns]
        .rank(method="average", pct=True)
        - 0.5
    )
    return out


if retchar_raw is not None:
    stock_processed = rank_normalize_cross_section(
        stock_clean,
        DATE_COL,
        characteristic_cols,
    )

    if characteristic_cols:
        min_rank = stock_processed[characteristic_cols].min().min()
        max_rank = stock_processed[characteristic_cols].max().max()
        print(f"Characteristic range after normalization: [{min_rank:.6f}, {max_rank:.6f}]")

        normalization_check = stock_processed.groupby(DATE_COL)[characteristic_cols].median().abs().stack()
        print("Median absolute monthly characteristic median:",
              f"{normalization_check.median():.6f}")

    display(stock_processed.head())
else:
    print("Skipped: stock panel has not been loaded.")



## 10. Prepare the macroeconomic panel

The paper uses a high-dimensional macroeconomic information set. The original transformations are series-specific because many macro series must be made stationary.

For this first replication notebook:

- If `Macro.csv` is the processed file distributed for the paper/project, we preserve its values.
- We standardize its date field and verify time coverage.
- We **do not** apply a generic transformation to all macro variables because that would not faithfully reproduce the paper's series-specific transformations.

If we later discover that the macro file contains raw rather than transformed series, the transformation map should be implemented explicitly in a separate preprocessing function.


In [ ]:

# 10A. Clean and inspect macro data

if macro_raw is not None:
    macro = clean_column_names(macro_raw)

    MACRO_DATE_COL = detect_column(
        macro.columns,
        ["date", "yyyymm", "month", "caldt", "time"],
        "macro date",
    )
    macro[MACRO_DATE_COL] = parse_monthly_date(macro[MACRO_DATE_COL])

    macro_feature_cols = [
        c for c in macro.columns
        if c != MACRO_DATE_COL and pd.api.types.is_numeric_dtype(macro[c])
    ]

    print("Macro date column :", MACRO_DATE_COL)
    print("Macro features    :", len(macro_feature_cols))
    print("Date range        :", macro[MACRO_DATE_COL].min(), "to", macro[MACRO_DATE_COL].max())
    print("Duplicate months  :", macro.duplicated([MACRO_DATE_COL]).sum())

    if len(macro_feature_cols) != 178:
        print(
            f"NOTE: detected {len(macro_feature_cols)} numeric macro features. "
            "The full paper information set contains 178 macroeconomic series. "
            "Confirm whether this file is already reduced/processed or includes nonnumeric fields."
        )

    display(macro.head())
else:
    print("Skipped: macro panel has not been loaded.")


In [ ]:

# 10B. Macro missingness report

if macro_raw is not None:
    macro_missing = (
        macro[macro_feature_cols]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_fraction")
        .to_frame()
    )

    print("Macro rows:", f"{len(macro):,}")
    print("Macro columns:", f"{macro.shape[1]:,}")
    display(macro_missing.head(25))
else:
    print("Skipped: macro panel has not been loaded.")



## 11. Fixed chronological train / validation / test split

A random split would leak future information into the past and would not reproduce the paper.

We use:

| Sample | Period |
|---|---|
| Training | 1967-01 through 1986-12 |
| Validation | 1987-01 through 1991-12 |
| Test | 1992-01 through 2016-12 |

These dates are kept fixed throughout the project.


In [ ]:

# 11A. Split helper

def chronological_split(df: pd.DataFrame, date_col: str):
    train = df[df[date_col].between(TRAIN_START, TRAIN_END)].copy()
    valid = df[df[date_col].between(VALID_START, VALID_END)].copy()
    test  = df[df[date_col].between(TEST_START, TEST_END)].copy()
    return train, valid, test


def split_summary(name, train, valid, test, date_col, id_col=None):
    rows = []
    for split_name, part in [("train", train), ("validation", valid), ("test", test)]:
        row = {
            "dataset": name,
            "split": split_name,
            "rows": len(part),
            "months": part[date_col].nunique() if len(part) else 0,
            "start": part[date_col].min() if len(part) else pd.NaT,
            "end": part[date_col].max() if len(part) else pd.NaT,
        }
        if id_col is not None:
            row["unique_stocks"] = part[id_col].nunique()
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:

# 11B. Apply the split

summaries = []

if retchar_raw is not None:
    stock_train, stock_valid, stock_test = chronological_split(
        stock_processed, DATE_COL
    )
    summaries.append(
        split_summary(
            "stock_characteristics",
            stock_train, stock_valid, stock_test,
            DATE_COL, ID_COL
        )
    )

if macro_raw is not None:
    macro_train, macro_valid, macro_test = chronological_split(
        macro, MACRO_DATE_COL
    )
    summaries.append(
        split_summary(
            "macro",
            macro_train, macro_valid, macro_test,
            MACRO_DATE_COL
        )
    )

if summaries:
    display(pd.concat(summaries, ignore_index=True))
else:
    print("No datasets loaded; split step skipped.")


In [ ]:

# 12. Leakage and consistency checks

def assert_disjoint_dates(train, valid, test, date_col):
    tr = set(train[date_col].dropna().unique())
    va = set(valid[date_col].dropna().unique())
    te = set(test[date_col].dropna().unique())

    assert tr.isdisjoint(va), "Training and validation dates overlap."
    assert tr.isdisjoint(te), "Training and test dates overlap."
    assert va.isdisjoint(te), "Validation and test dates overlap."


if retchar_raw is not None:
    assert_disjoint_dates(stock_train, stock_valid, stock_test, DATE_COL)
    assert not stock_processed.duplicated([DATE_COL, ID_COL]).any(), (
        "Duplicate stock-month observations remain. Investigate before modeling."
    )
    print("✓ Stock split dates are disjoint.")
    print("✓ No duplicate stock-month rows.")

if macro_raw is not None:
    assert_disjoint_dates(macro_train, macro_valid, macro_test, MACRO_DATE_COL)
    print("✓ Macro split dates are disjoint.")

if retchar_raw is not None and macro_raw is not None:
    stock_months = set(stock_processed[DATE_COL].dropna().unique())
    macro_months = set(macro[MACRO_DATE_COL].dropna().unique())

    shared = stock_months & macro_months
    print(f"Shared stock/macro months: {len(shared):,}")
    print(f"Stock-only months        : {len(stock_months - macro_months):,}")
    print(f"Macro-only months        : {len(macro_months - stock_months):,}")



## 13. Save processed outputs

The processed files below are **generated artifacts**, so they belong in `data/processed/`, which is ignored by Git in this repository.

We save Parquet files because they preserve dtypes and are substantially faster and smaller than CSV for large stock panels.


In [ ]:

# 13A. Save only when the source datasets are available

if retchar_raw is not None:
    stock_out = PROCESSED_DIR / "retchar_processed.parquet"
    stock_processed.to_parquet(stock_out, index=False)
    print("Saved:", stock_out)

if macro_raw is not None:
    macro_out = PROCESSED_DIR / "macro_processed.parquet"
    macro.to_parquet(macro_out, index=False)
    print("Saved:", macro_out)


In [ ]:

# 13B. Optional split-specific outputs

if retchar_raw is not None:
    stock_train.to_parquet(PROCESSED_DIR / "retchar_train.parquet", index=False)
    stock_valid.to_parquet(PROCESSED_DIR / "retchar_validation.parquet", index=False)
    stock_test.to_parquet(PROCESSED_DIR / "retchar_test.parquet", index=False)

if macro_raw is not None:
    macro_train.to_parquet(PROCESSED_DIR / "macro_train.parquet", index=False)
    macro_valid.to_parquet(PROCESSED_DIR / "macro_validation.parquet", index=False)
    macro_test.to_parquet(PROCESSED_DIR / "macro_test.parquet", index=False)

print("Split-specific save step complete.")



## 14. Replication checklist

Before moving to the next notebook, verify:

- [ ] `RetChar.csv` loads successfully.
- [ ] `Macro.csv` loads successfully.
- [ ] The stock panel contains the intended 46 firm characteristics.
- [ ] The macro panel is confirmed to represent the intended processed information set.
- [ ] The stock sample covers the intended 1967–2016 period.
- [ ] Complete-case filtering is consistent with the paper replication design.
- [ ] Characteristics are rank-normalized cross-sectionally.
- [ ] Training, validation, and test dates are non-overlapping.
- [ ] Processed Parquet files have been written to `data/processed/`.
- [ ] Any difference from the authors' data construction is documented before modeling.

## Next notebook

**`02_exploratory_analysis.ipynb`**

The next notebook should verify sample sizes, return distributions, characteristic distributions, time coverage, cross-sectional counts, and macroeconomic variation before any model is estimated.
